# 03 — Модель

Цель: обогнать бейзлайн (общая тепловая карта, lift ×1,38 на новых боях), учитывая класс и уровень танка,
место в списке и состав боя — но не дробя данные на кусочки, как карта по классам.

**Модель** — `HistGradientBoostingRegressor` из scikit-learn: градиентный бустинг на деревьях, устроенный как LightGBM.
Она предсказывает `damage_rel` по контексту и зоне. **Совет**: для танка перебираем все зоны карты,
предсказываем урон в каждой и берём 3 лучшие.

Проверка — та же, что у бейзлайна: lift на кросс-валидации старых боёв и один раз на новых.

## 1. Загрузка

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

DATA = Path("../data/processed")
features = pd.read_parquet(DATA / "features.parquet")
frozen_ids = pd.read_csv(DATA / "train_battles.csv")["arena_id"]

dev = features[features["arena_id"].isin(frozen_ids)].copy()   # старые бои: обучение и подбор
new = features[~features["arena_id"].isin(frozen_ids)].copy()  # новые бои: только финальная проверка
print("старых боёв:", dev["arena_id"].nunique(), "| новых:", new["arena_id"].nunique())

старых боёв: 3485 | новых: 590


## 2. Насколько часто бывают «экстремальные» составы

Сколько в команде ПТ, ТТ и арты. Если боёв с 8+ ПТ единицы, модель про них знает мало.

In [2]:
pd.concat({
    "ПТ (ally_n_td)": dev.groupby("arena_id")["ally_n_td"].first().value_counts().sort_index(),
    "ТТ (ally_n_ht)": dev.groupby("arena_id")["ally_n_ht"].first().value_counts().sort_index(),
    "арта (ally_n_spg)": dev.groupby("arena_id")["ally_n_spg"].first().value_counts().sort_index(),
}, axis=1).fillna(0).astype(int)

,ПТ (ally_n_td),ТТ (ally_n_ht),арта (ally_n_spg)
0,40,18,1453
1,141,44,1344
2,466,136,688
3,725,354,0
4,857,770,0
5,1192,942,0
6,62,674,0
7,2,350,0
8,0,123,0
9,0,49,0


## 3. Признаки

- **Категории** (карта, сторона, класс) — модель сама разбивает их на группы.
- **`zone_prior`** — сглаженное среднее `damage_rel` зоны из тепловой карты (target encoding).
  Модель начинает с того, что знает бейзлайн, и учится поправкам к нему.
- **Зона** — ещё и двумя числами `zone_x`, `zone_z`: так модель понимает, что соседние зоны похожи.
- **Контекст**: уровень, место в списке, состав своей команды. Состав врага не берём:
  в 96,7% боёв он совпадает с нашим и ничего не добавляет.

Категориям нужен один и тот же список значений при обучении и предсказании,
поэтому берём его по всей таблице.

In [3]:
CATEGORICAL = ["map_name", "team", "tank_class"]
NUMERIC = ["zone_prior", "zone_x", "zone_z", "tank_tier", "tier_diff",
           "ally_n_lt", "ally_n_mt", "ally_n_ht", "ally_n_td", "ally_n_spg", "ally_avg_tier"]
FEATURES = CATEGORICAL + NUMERIC
TARGET = "damage_rel"

CATEGORIES = {col: sorted(features[col].astype(str).unique()) for col in CATEGORICAL}


def to_X(rows):
    """Таблица признаков для модели: категории — в тип category с общим списком значений."""
    X = rows[FEATURES].copy()
    for col in CATEGORICAL:
        X[col] = pd.Categorical(X[col].astype(str), categories=CATEGORIES[col])
    return X


def train_model(train):
    model = HistGradientBoostingRegressor(
        categorical_features="from_dtype",
        max_iter=300,
        learning_rate=0.05,
        min_samples_leaf=50,       # не меньше 50 танков в листе дерева — защита от выучивания случайностей
        early_stopping=True,
        random_state=42,
    )
    train = add_zone_prior_oof(train)
    model.fit(to_X(train), train[TARGET])
    return model

## 4. Совет модели и бейзлайна

**Кандидаты** — зоны, где в обучении на этой карте за эту сторону было хотя бы `MIN_COUNT` танков
(как у бейзлайна, 10). Советовать зону, в которой почти никто не бывал, нельзя: про неё нет данных.

**Совет модели**: для каждого танка из проверки подставляем по очереди все зоны-кандидаты,
предсказываем `damage_rel` и берём 3 лучшие. Совет у каждого танка **свой**: он зависит от его класса и состава боя.

**Совет бейзлайна**: 3 зоны с наибольшим сглаженным средним — одни и те же для всех танков на этой карте и стороне.

**Lift** считается одинаково: средний `damage_rel` тех, кто реально оказался в одной из 3 советуемых зон,
делённый на средний у остальных.

In [4]:
MIN_COUNT = 10
K = 50
ZONE_KEYS = ["map_name", "team", "zone"]


def zone_stats(train, k=K):
    """Статистика зон по обучающим строкам: координаты, число танков и сглаженное среднее."""
    stats = train.groupby(ZONE_KEYS).agg(
        zone_x=("zone_x", "first"), zone_z=("zone_z", "first"),
        zone_sum=(TARGET, "sum"), zone_count=(TARGET, "count"),
    ).reset_index()
    stats["zone_prior"] = (stats["zone_sum"] + k) / (stats["zone_count"] + k)
    return stats


def add_zone_prior(rows, train):
    """Приклеить к строкам zone_prior, посчитанный ТОЛЬКО по train. Новая зона — 1.0."""
    out = rows.merge(zone_stats(train)[ZONE_KEYS + ["zone_prior"]], on=ZONE_KEYS, how="left")
    out["zone_prior"] = out["zone_prior"].fillna(1.0)
    return out


def add_zone_prior_oof(train, n_parts=5, seed=0):
    """zone_prior для самих обучающих строк: для каждой части — по остальным частям (out-of-fold).
    Иначе строка «видела бы» свой собственный урон в среднем зоны."""
    battles = train["arena_id"].unique().copy()
    np.random.default_rng(seed).shuffle(battles)
    pieces = []
    for part in np.array_split(battles, n_parts):
        inside = train["arena_id"].isin(part)
        pieces.append(add_zone_prior(train[inside], train[~inside]))
    return pd.concat(pieces, ignore_index=True)


def candidate_zones(train, min_count=MIN_COUNT):
    stats = zone_stats(train)
    return stats[stats["zone_count"] >= min_count]


def lift_of(checked):
    in_top = checked["in_top"]
    total = checked.loc[in_top, TARGET].mean() / checked.loc[~in_top, TARGET].mean()
    by_class = checked.groupby(["tank_class", "in_top"])[TARGET].mean().unstack().reindex(columns=[False, True])
    return total, by_class[True] / by_class[False]


def baseline_check(train, test):
    cand = candidate_zones(train)
    top = cand.sort_values("zone_prior", ascending=False).groupby(["map_name", "team"]).head(3)
    checked = test.merge(top[ZONE_KEYS].assign(in_top=True), on=ZONE_KEYS, how="left")
    checked["in_top"] = checked["in_top"].fillna(False).astype(bool)
    return checked


def model_check(model, train, test, n=3):
    cand = candidate_zones(train)[["map_name", "team", "zone", "zone_x", "zone_z", "zone_prior"]]
    rows = test.reset_index(drop=True).copy()
    rows["row_id"] = rows.index

    # каждый танк × все зоны-кандидаты его карты и стороны
    grid = rows.drop(columns=["zone", "zone_x", "zone_z", "zone_prior"], errors="ignore").merge(cand, on=["map_name", "team"])
    grid["pred"] = model.predict(to_X(grid))
    top = grid.sort_values("pred", ascending=False).groupby("row_id").head(n)

    chosen = set(zip(top["row_id"], top["zone"]))
    rows["in_top"] = [(r, z) in chosen for r, z in zip(rows["row_id"], rows["zone"])]
    return rows

## 5. Кросс-валидация: модель против бейзлайна

5 разделений старых боёв по боям. На каждом — обучаем, советуем, считаем lift для обоих.
Занимает несколько минут.

In [5]:
N_FOLDS = 5
battles = dev["arena_id"].unique().copy()
np.random.default_rng(42).shuffle(battles)
folds = np.array_split(battles, N_FOLDS)

cv_rows = []
for i, fold in enumerate(folds, start=1):
    is_test = dev["arena_id"].isin(fold)
    train, test = dev[~is_test], dev[is_test]

    base_total, base_cls = lift_of(baseline_check(train, test))
    model = train_model(train)
    model_total, model_cls = lift_of(model_check(model, train, test))

    cv_rows.append({"fold": i, "бейзлайн": base_total, "модель": model_total,
                    **{f"база {c}": v for c, v in base_cls.items()},
                    **{f"модель {c}": v for c, v in model_cls.items()}})
    print(f"фолд {i}: бейзлайн {base_total:.3f} | модель {model_total:.3f}")

cv = pd.DataFrame(cv_rows)
print()
print(f"бейзлайн: {cv['бейзлайн'].mean():.3f} ± {cv['бейзлайн'].std():.3f}")
print(f"модель:   {cv['модель'].mean():.3f} ± {cv['модель'].std():.3f}")

фолд 1: бейзлайн 1.338 | модель 1.321
фолд 2: бейзлайн 1.328 | модель 1.297
фолд 3: бейзлайн 1.319 | модель 1.294
фолд 4: бейзлайн 1.343 | модель 1.304
фолд 5: бейзлайн 1.297 | модель 1.265

бейзлайн: 1.325 ± 0.018
модель:   1.296 ± 0.020


In [6]:
# lift по классам: среднее ± разброс по фолдам
per_class = {}
for cls in sorted(dev["tank_class"].unique()):
    b, m = cv[f"база {cls}"], cv[f"модель {cls}"]
    per_class[cls] = {"бейзлайн": f"{b.mean():.2f} ± {b.std():.2f}", "модель": f"{m.mean():.2f} ± {m.std():.2f}"}
pd.DataFrame(per_class).T

,бейзлайн,модель
AT-SPG,1.30 ± 0.05,1.29 ± 0.02
SPG,0.90 ± 0.35,1.06 ± 0.44
heavyTank,1.29 ± 0.05,1.24 ± 0.04
lightTank,1.34 ± 0.10,1.32 ± 0.10
mediumTank,1.26 ± 0.03,1.24 ± 0.02


## 6. Финальная проверка на новых боях

Запускать **один раз**, когда в разделах 3–5 всё выбрано. Ничего не подбираем по этим цифрам.

In [7]:
final_model = train_model(dev)

base_total, base_cls = lift_of(baseline_check(dev, new))
model_total, model_cls = lift_of(model_check(final_model, dev, new))

print(f"новые бои ({new['arena_id'].nunique()}): бейзлайн lift {base_total:.3f} | модель lift {model_total:.3f}")
pd.DataFrame({"бейзлайн": base_cls, "модель": model_cls}).round(2)

новые бои (590): бейзлайн lift 1.384 | модель lift 1.342


,бейзлайн,модель
tank_class,,
AT-SPG,1.38,1.42
SPG,0.59,0.69
heavyTank,1.34,1.30
lightTank,1.18,0.94
mediumTank,1.34,1.25


## 7. Что модель советует: много тяжей против мало тяжей

ПТ в командах почти всегда 2–5, арты 0–2 — балансировщик их ограничивает. А число **тяжей** меняется от 0 до 11:
это главное, чем бои различаются по составу.

ТТ 10 уровня в топе списка, один и тот же бой, меняем только число ТТ в команде (у врага почти всегда столько же).
Меняется ли совет?

In [8]:
MAP, TEAM, CLASS = "05_prohorovka", 1, "heavyTank"

example = dev[(dev["map_name"] == MAP) & (dev["team"] == TEAM) & (dev["tank_class"] == CLASS)].iloc[[0]].copy()
cand = candidate_zones(dev)
cand = cand[(cand["map_name"] == MAP) & (cand["team"] == TEAM)][["map_name", "team", "zone", "zone_x", "zone_z", "zone_prior"]]

for n_ht in [1, 3, 5, 7, 9]:
    row = example.copy()
    row["ally_n_ht"] = n_ht
    row["tank_tier"], row["tier_diff"] = 10, 0
    grid = row.drop(columns=["zone", "zone_x", "zone_z", "zone_prior"], errors="ignore").merge(cand, on=["map_name", "team"])
    grid["pred"] = final_model.predict(to_X(grid))
    best = grid.sort_values("pred", ascending=False).head(3)
    print(f"ТТ в команде: {n_ht} → зоны {list(best['zone'])}, ожидаемый damage_rel {best['pred'].round(2).tolist()}")

ТТ в команде: 1 → зоны ['-2_0', '-5_2', '4_-1'], ожидаемый damage_rel [1.41, 1.3, 1.23]
ТТ в команде: 3 → зоны ['-2_0', '-5_2', '-3_0'], ожидаемый damage_rel [1.43, 1.32, 1.24]
ТТ в команде: 5 → зоны ['-2_0', '-5_2', '-3_0'], ожидаемый damage_rel [1.44, 1.32, 1.23]
ТТ в команде: 7 → зоны ['-2_0', '-5_2', '-3_0'], ожидаемый damage_rel [1.44, 1.32, 1.23]
ТТ в команде: 9 → зоны ['-2_0', '-5_2', '-3_0'], ожидаемый damage_rel [1.44, 1.32, 1.23]
